In [ ]:
import os
import re
import pandas as pd
from histcite.process_file import ProcessFile
from histcite.compute_metrics import ComputeMetrics
from histcite.network_graph import GraphViz

In [ ]:
# 读取文件，去重，识别引文关系
folder_path = 'dataset/wos'
source_type = 'wos'

# folder_path = 'dataset/cssci'
# source_type = 'cssci'

# folder_path = 'dataset/scopus'
# source_type = 'scopus'

process = ProcessFile(folder_path,source_type)
process.concat_table() # 合并多个文件
process.process_citation() # 识别引用关系
docs_table = process.docs_table
reference_table = process.reference_table
docs_table.head()

In [ ]:
reference_table.head()

In [ ]:
# 基本描述统计
cm = ComputeMetrics(docs_table, reference_table, source_type)
cm.write2excel(os.path.join(folder_path,'result','statistics.xlsx'))

图文件可以使用 [Graphviz在线编辑器](http://magjac.com/graphviz-visual-editor/) 或本地的 [Graphviz工具](https://graphviz.org/) 生成引文网络图。  

In [ ]:
# 生成引文网络图文件

# 选取LSC最高的100篇文献
doc_indices = docs_table.sort_values('LCS', ascending=False).index[:100]
# 选取LSC大于5的文献
# doc_indices = docs_table[citation_table['LCS']>5].index

graph = GraphViz(docs_table, source_type)
graph_dot_file = graph.generate_dot_file(doc_indices,allow_external_node=False)

# 保存graph.dot文件
with open(os.path.join(folder_path,'result','graph.dot'),'w') as f:
    f.write(graph_dot_file)
print(graph_dot_file)

In [ ]:
# 导出图节点文件
graph_node_file = graph.generate_graph_node_file()
graph_node_file.to_excel(os.path.join(folder_path,'result','graph.node.xlsx'),index=False)
graph_node_file